# Example 3:Ridge Regression
This example try to show how to use our PLQ Composite Decomposition tool to perform Ridge Regression.

# 1. Data Generation

In [1]:
from plqcom import PLQLoss, plq_to_rehloss, affine_transformation
import numpy as np
from rehline import ReHLine

n_samples, n_features = 1000, 5
rng = np.random.RandomState(0)
X = rng.randn(n_samples, n_features)
beta = rng.randn(n_features)
y = np.dot(X, beta) + rng.normal(scale=0.1, size=n_samples)

# 2.Create and Decompose the PLQ Loss

In [2]:
plqloss = PLQLoss(quad_coef={'a': np.array([1.]), 'b': np.array([0.]), 'c': np.array([0.])},
                  cutpoints=np.array([]))

In [3]:
rehloss = plq_to_rehloss(plqloss)

In [4]:
rehloss.rehu_cut, rehloss.rehu_coef, rehloss.rehu_intercept

(array([[inf],
        [inf]]),
 array([[-1.41421356],
        [ 1.41421356]]),
 array([[-0.],
        [ 0.]]))

# 3. Broadcast to all samples

In [5]:
rehloss = affine_transformation(rehloss, n=X.shape[0], c=1, p=-1, q=y)

# 4. Solve with ReHLine

ReHLine (>=0.1.0) supports two calling styles:

- **4a. Low-level API** — After plqcom decomposition (steps 1–3), pass `rehloss` parameters via `_Tau`, `_S`, `_T`, etc. Use this for **custom** PLQ losses from plqcom.
- **4b. Scikit-learn style** — For **built-in** PLQ losses (MSE, Huber, MAE, ...), use `plq_Ridge_Regressor` with `fit(X, y)` directly.

## 4a. Low-level API (with plqcom decomposition)

In [6]:
clf = ReHLine(C=1)
clf._Tau, clf._S, clf._T = rehloss.rehu_cut, rehloss.rehu_coef, rehloss.rehu_intercept
clf.fit(X=X)
print('sol provided by rehline: %s' % clf.coef_)

sol provided by rehline: [ 0.31029299 -0.73855141 -1.53883908 -0.56076169 -1.6047202 ]


## 4b. Scikit-learn style (built-in MSE loss)

For ridge regression with squared error, use `plq_Ridge_Regressor` with the built-in `MSE` loss:

In [ ]:
from rehline import plq_Ridge_Regressor

clf_sk = plq_Ridge_Regressor(loss={'name': 'MSE'}, C=1)
clf_sk.fit(X, y)
print('sol provided by plq_Ridge_Regressor: %s' % clf_sk.coef_)

Compare with the solution provided by sklearn

In [7]:
from sklearn.linear_model import Ridge
clf1 = Ridge(alpha=0.5)
clf1.fit(X, y)
print('sol provided by sklearn: %s' % clf1.coef_)

sol provided by sklearn: [ 0.31017419 -0.73849146 -1.53893293 -0.56084128 -1.60476756]
